In [4]:
### CELL 1: Импорты, seed и среда ###
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Создаем папку для артефактов
os.makedirs("artifacts", exist_ok=True)

def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False

FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")
if FAISS_READY:
    import faiss
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print(f"FAISS доступен: {FAISS_READY}")
print(f"Устройство: {DEVICE}")

FAISS доступен: True
Устройство: cuda


In [2]:
### CELL 2: База знаний ###
documents = [
    {"doc_id": "ml_01", "title": "Машинное обучение", "text": "Машинное обучение (ML) — это класс методов искусственного интеллекта, характерной чертой которых является не прямое решение задачи, а обучение за счёт применения решений множества сходных задач. Для этого используются алгоритмы поиска закономерностей в данных."},
    {"doc_id": "ml_02", "title": "Обучение с учителем", "text": "Обучение с учителем (Supervised learning) — один из разделов машинного обучения. Модель обучается на размеченном наборе данных, где для каждого примера известен правильный ответ (целевая переменная). Типичные задачи: классификация и регрессия."},
    {"doc_id": "ml_03", "title": "Обучение без учителя", "text": "Обучение без учителя (Unsupervised learning) используется, когда данные не имеют разметки. Алгоритм сам должен найти внутреннюю структуру или скрытые закономерности в данных. Популярная задача — кластеризация (например, алгоритм K-means)."},
    {"doc_id": "ml_04", "title": "Линейная регрессия", "text": "Линейная регрессия — базовая модель для прогнозирования непрерывной величины. Она находит линейную зависимость между входными признаками и целевой переменной, минимизируя среднеквадратичную ошибку (MSE) предсказаний на обучающей выборке."},
    {"doc_id": "ml_05", "title": "Деревья решений", "text": "Деревья решений — алгоритм, который разбивает пространство признаков на области с помощью последовательности простых правил (if-then). Они легко интерпретируются человеком, но склонны к сильному переобучению на сложных данных."},
    {"doc_id": "ml_06", "title": "Переобучение", "text": "Переобучение (Overfitting) возникает, когда модель слишком хорошо запоминает обучающую выборку, включая её шум, и теряет способность обобщать данные. На тестовой выборке такая модель показывает низкое качество. Способы борьбы: регуляризация, уменьшение сложности модели."},
    {"doc_id": "ml_07", "title": "Кросс-валидация", "text": "Кросс-валидация — метод оценки качества модели на ограниченном объеме данных. Выборка разбивается на K фолдов (частей). Модель обучается K раз на разных комбинациях обучающих и тестовых фолдов, что дает более надежную оценку качества."},
    {"doc_id": "ml_08", "title": "Градиентный спуск", "text": "Градиентный спуск — итерационный алгоритм оптимизации. Он используется для поиска минимума функции потерь путем движения в направлении, антипараллельном градиенту функции. Размер шага регулируется параметром learning rate (скорость обучения)."},
    {"doc_id": "ml_09", "title": "Нейронные сети", "text": "Искусственные нейронные сети состоят из слоев связанных нейронов. Они способны извлекать сложные нелинейные признаки из сырых данных (например, пикселей изображений). Глубокое обучение (Deep Learning) использует сети с большим числом скрытых слоев."},
    {"doc_id": "ml_10", "title": "Метрики классификации", "text": "Для оценки задач классификации недостаточно метрики Accuracy (доля правильных ответов), особенно при дисбалансе классов. Часто используют Precision (точность), Recall (полнота) и агрегированную метрику F1-score, а также ROC-AUC."}
]

docs_df = pd.DataFrame(documents)
print(f"Число исходных документов: {len(docs_df)}")
display(docs_df.head(3))
# Комментарий: База знаний посвящена основам ML. Тексты компактные, с четкой терминологией, что идеально для mini-RAG.

Число исходных документов: 10


,doc_id,title,text
0,ml_01,Машинное обучение,Машинное обучение (ML) — это класс методов иск...
1,ml_02,Обучение с учителем,Обучение с учителем (Supervised learning) — од...
2,ml_03,Обучение без учителя,Обучение без учителя (Unsupervised learning) и...


In [3]:
### CELL 3: Чанкинг ###
def chunk_text(text: str, chunk_size: int = 15, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if chunk_words:
            chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks

# Демонстрация
sample_chunks = chunk_text(documents[5]["text"], chunk_size=15, overlap=5)
print(f"Оригинал: {documents[5]['text']}\n")
for i, c in enumerate(sample_chunks):
    print(f"Чанк {i+1}: {c}")

Оригинал: Переобучение (Overfitting) возникает, когда модель слишком хорошо запоминает обучающую выборку, включая её шум, и теряет способность обобщать данные. На тестовой выборке такая модель показывает низкое качество. Способы борьбы: регуляризация, уменьшение сложности модели.

Чанк 1: Переобучение (Overfitting) возникает, когда модель слишком хорошо запоминает обучающую выборку, включая её шум, и теряет
Чанк 2: включая её шум, и теряет способность обобщать данные. На тестовой выборке такая модель показывает низкое
Чанк 3: выборке такая модель показывает низкое качество. Способы борьбы: регуляризация, уменьшение сложности модели.


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 13987.81it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: SentenceTransformer (MiniLM). Всего чанков: 30


,doc_id,title,chunk_text,chunk_id,score
18,ml_07,Кросс-валидация,Кросс-валидация — метод оценки качества модели...,ml_07_0,0.717779
20,ml_07,Кросс-валидация,на разных комбинациях обучающих и тестовых фол...,ml_07_2,0.489531
28,ml_10,Метрики классификации,особенно при дисбалансе классов. Часто использ...,ml_10_1,0.355379


Базовый hit@3 = recall@3 = 0.90
Hit@3 при chunk_size=30: 0.70


,query,before_retrieved_sources,after_retrieved_sources,changed
0,Что максимизирует агент? | В чем суть механизм...,"ml_01,ml_08,ml_06","ml_11,ml_11,ml_11",True
1,Что максимизирует агент? | В чем суть механизм...,"ml_01,ml_05,ml_06","ml_12,ml_01,ml_05",True


,question,answer,retrieved_sources
0,Что такое машинное обучение?,Машинное обучение (ML) — это класс методов иск...,"ml_02,ml_01,ml_11"
1,Для чего используют кросс-валидацию?,Кросс-валидация — метод оценки качества модели...,"ml_07,ml_07,ml_10"
2,Что такое механизм Self-Attention?,"Трансформеры — архитектура нейросетей, основан...","ml_12,ml_01,ml_05"
3,Какой алгоритм лучше использовать для кластери...,Для этого используются алгоритмы поиска законо...,"ml_03,ml_01,ml_03"
4,Как заварить чай?,произвела революцию в NLP и стала основой для ...,"ml_01,ml_08,ml_12"


In [5]:
### CELL 4: Эмбеддинги и FAISS ###
class Embedder:
    def __init__(self, device="cpu"):
        self.is_dense = False
        if SENTENCE_TRANSFORMERS_READY:
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
                self.is_dense = True
                self.name = "SentenceTransformer (MiniLM)"
            except:
                pass
        if not self.is_dense:
            self.model = TfidfVectorizer(ngram_range=(1,2))
            self.name = "TF-IDF (Fallback)"

    def encode(self, texts, is_fit=False):
        if self.is_dense:
            return self.model.encode(texts, normalize_embeddings=True).astype("float32")
        else:
            if is_fit:
                vecs = self.model.fit_transform(texts).toarray().astype("float32")
            else:
                vecs = self.model.transform(texts).toarray().astype("float32")
            norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
            return vecs / norms

@dataclass
class Indexer:
    df: pd.DataFrame
    embedder: Embedder
    index: object

def build_index(docs, chunk_size=15, overlap=5, embedder=None) -> Indexer:
    rows = []
    for d in docs:
        for i, chunk in enumerate(chunk_text(d["text"], chunk_size, overlap)):
            rows.append({"doc_id": d["doc_id"], "title": d["title"], "chunk_text": chunk, "chunk_id": f"{d['doc_id']}_{i}"})
    df = pd.DataFrame(rows)
    
    vecs = embedder.encode(df["chunk_text"].tolist(), is_fit=True)
    if FAISS_READY:
        idx = faiss.IndexFlatIP(vecs.shape[1])
        idx.add(vecs)
    else:
        idx = vecs # Fallback dummy
    return Indexer(df, embedder, idx)

def search(query: str, indexer: Indexer, top_k: int = 3):
    q_vec = indexer.embedder.encode([query], is_fit=False)
    if FAISS_READY:
        scores, ids = indexer.index.search(q_vec, top_k)
        scores, ids = scores[0], ids[0]
    else:
        sim = (indexer.index @ q_vec.T).reshape(-1)
        ids = np.argsort(-sim)[:top_k]
        scores = sim[ids]
    
    res = indexer.df.iloc[ids].copy()
    res["score"] = scores
    return res

embedder = Embedder(DEVICE)
indexer = build_index(documents, embedder=embedder)
print(f"Модель: {embedder.name}. Всего чанков: {len(indexer.df)}")

# Проверка
display(search("Что такое кросс-валидация?", indexer))

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 13084.80it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: SentenceTransformer (MiniLM). Всего чанков: 30


,doc_id,title,chunk_text,chunk_id,score
18,ml_07,Кросс-валидация,Кросс-валидация — метод оценки качества модели...,ml_07_0,0.717779
20,ml_07,Кросс-валидация,на разных комбинациях обучающих и тестовых фол...,ml_07_2,0.489531
28,ml_10,Метрики классификации,особенно при дисбалансе классов. Часто использ...,ml_10_1,0.355379


In [6]:
### CELL 5: Контрольные запросы и оценка ###
test_queries = [
    {"query": "Что такое машинное обучение?", "expected_doc": "ml_01"},
    {"query": "Для чего нужна разметка данных?", "expected_doc": "ml_02"},
    {"query": "Алгоритм K-means это обучение с учителем?", "expected_doc": "ml_03"},
    {"query": "Что минимизирует линейная регрессия?", "expected_doc": "ml_04"},
    {"query": "Минусы деревьев решений", "expected_doc": "ml_05"},
    {"query": "Как бороться с переобучением?", "expected_doc": "ml_06"},
    {"query": "На сколько фолдов разбивается выборка?", "expected_doc": "ml_07"},
    {"query": "Что такое learning rate?", "expected_doc": "ml_08"},
    {"query": "Глубокое обучение и слои нейронов", "expected_doc": "ml_09"},
    {"query": "Что такое F1-score?", "expected_doc": "ml_10"}
]

def evaluate(queries, idxer, top_k=3):
    results = []
    for q in queries:
        res_df = search(q["query"], idxer, top_k)
        retrieved_docs = res_df["doc_id"].tolist()
        hit = int(q["expected_doc"] in retrieved_docs)
        results.append({
            "query": q["query"],
            "expected_source": q["expected_doc"],
            "retrieved_sources": ",".join(retrieved_docs),
            "hit_at_k": hit
        })
    eval_df = pd.DataFrame(results)
    return eval_df

eval_base = evaluate(test_queries, indexer)
eval_base.to_csv("artifacts/retrieval_eval.csv", index=False)
hit_k = eval_base["hit_at_k"].mean()
print(f"Базовый hit@3 = recall@3 = {hit_k:.2f}")

Базовый hit@3 = recall@3 = 0.90


In [7]:
### CELL 6: Эксперимент по параметрам (chunk_size) ###
indexer_large = build_index(documents, chunk_size=30, overlap=10, embedder=embedder)
eval_large = evaluate(test_queries, indexer_large)
print(f"Hit@3 при chunk_size=30: {eval_large['hit_at_k'].mean():.2f}")

Hit@3 при chunk_size=30: 0.70


In [8]:
### CELL 7: Обновление базы знаний ###
new_docs = [
    {"doc_id": "ml_11", "title": "Обучение с подкреплением", "text": "Обучение с подкреплением (Reinforcement learning) — метод обучения, при котором агент взаимодействует со средой и получает награды (штрафы) за свои действия. Цель агента — максимизировать суммарную награду."},
    {"doc_id": "ml_12", "title": "Трансформеры", "text": "Трансформеры — архитектура нейросетей, основанная на механизме внимания (Self-Attention). Она произвела революцию в NLP и стала основой для больших языковых моделей (LLM), таких как GPT."},
]
all_docs = documents + new_docs
indexer_updated = build_index(all_docs, embedder=embedder)

# Сравнение до/после на новых запросах
upd_queries = [
    {"query": "Что максимизирует агент?", "expected_doc": "ml_11"},
    {"query": "В чем суть механизма Self-Attention?", "expected_doc": "ml_12"}
]

before_df = evaluate(upd_queries, indexer)
after_df = evaluate(upd_queries, indexer_updated)

comparison = pd.DataFrame({
    "query": upd_queries[0]["query"] + " | " + upd_queries[1]["query"],
    "before_retrieved_sources": before_df["retrieved_sources"].tolist(),
    "after_retrieved_sources": after_df["retrieved_sources"].tolist()
})
comparison["changed"] = comparison["before_retrieved_sources"] != comparison["after_retrieved_sources"]
comparison.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
display(comparison)

,query,before_retrieved_sources,after_retrieved_sources,changed
0,Что максимизирует агент? | В чем суть механизм...,"ml_01,ml_08,ml_06","ml_11,ml_11,ml_11",True
1,Что максимизирует агент? | В чем суть механизм...,"ml_01,ml_05,ml_06","ml_12,ml_01,ml_05",True


In [9]:
### CELL 8: Mini-RAG ###
def split_sentences(text: str):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def mini_rag(query: str, idxer: Indexer, top_k=3):
    res_df = search(query, idxer, top_k)
    context_chunks = res_df["chunk_text"].tolist()
    sources = res_df["doc_id"].tolist()
    
    # Очень простой extractive генератор (как в демо)
    sentences = []
    for c in context_chunks:
        sentences.extend(split_sentences(c))
    
    # TF-IDF для выбора лучшего предложения из контекста
    vec = TfidfVectorizer(ngram_range=(1,2)).fit(sentences + [query])
    mat = vec.transform(sentences + [query]).toarray()
    sims = (mat[:-1] @ mat[-1].T) / (np.linalg.norm(mat[:-1], axis=1) * np.linalg.norm(mat[-1]) + 1e-9)
    best_idx = np.argsort(-sims)[:2] # берем 2 лучших предложения
    answer = " ".join([sentences[i] for i in best_idx if sims[i] > 0])
    
    if not answer:
        answer = "Не удалось найти точный ответ в контексте."
        
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ",".join(sources)
    }

rag_questions = [
    "Что такое машинное обучение?",
    "Для чего используют кросс-валидацию?",
    "Что такое механизм Self-Attention?",
    "Какой алгоритм лучше использовать для кластеризации?",
    "Как заварить чай?" # Запрос вне домена
]

rag_results = [mini_rag(q, indexer_updated) for q in rag_questions]
rag_df = pd.DataFrame(rag_results)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)
display(rag_df)

,question,answer,retrieved_sources
0,Что такое машинное обучение?,Машинное обучение (ML) — это класс методов иск...,"ml_02,ml_01,ml_11"
1,Для чего используют кросс-валидацию?,Кросс-валидация — метод оценки качества модели...,"ml_07,ml_07,ml_10"
2,Что такое механизм Self-Attention?,"Трансформеры — архитектура нейросетей, основан...","ml_12,ml_01,ml_05"
3,Какой алгоритм лучше использовать для кластери...,Для этого используются алгоритмы поиска законо...,"ml_03,ml_01,ml_03"
4,Как заварить чай?,произвела революцию в NLP и стала основой для ...,"ml_01,ml_08,ml_12"
